API 功能：

drawdown.plot(...)：pandas 的绘图 API，底层调用 matplotlib。

figsize=(10,4)：图像大小，宽 10 英寸，高 4 英寸。

(10,4) 是 Python 元组。

color="black"：线条颜色为黑色。

title="Drawdown Curve"：图表标题。

ax：matplotlib 的 Axes 对象，后续可以继续设置坐标轴、填充区域等。

语法：连续访问对象属性和方法：

ax.yaxis：取得 y 轴对象。

.set_major_formatter(...)：设置 y 轴主刻度显示格式。

mtick.PercentFormatter(1.0)：创建百分比格式化器。

API 功能：把 y 轴数值显示成百分比。因为 drawdown 的值是小数比例，这里的 1.0 表示：输入数据 1.0 对应 100%。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

df = pd.read_csv("../data/sample_etf_daily.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["symbol", "date"], ascending=[True, True])

for symbol, group in df.groupby("symbol"):
    group = group.sort_values("date").dropna(subset=["close"]).set_index("date")
    if group.empty:
        continue

    cummax = group["close"].cummax()
    drawdown = group["close"] / cummax - 1
    max_drawdown = drawdown.min()
    print(f"{symbol} max drawdown: {max_drawdown:.2%}")

    ax = drawdown.plot(
        figsize=(10, 4),
        color="black",
        title=f"{symbol} Drawdown Curve (Max Drawdown: {max_drawdown:.2%})",
    )
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.fill_between(
        drawdown.index,
        drawdown.values,
        0,
        where=drawdown.values < 0,
        color="red",
        alpha=0.25,
        interpolate=True,
    )
    plt.show()